# importing dataset

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("abhinavwalia95/entity-annotated-corpus")

print("Path to dataset files:", path)

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/abhinavwalia95/entity-annotated-corpus/versions/4


In [2]:
import pandas as pd
df = pd.read_csv(path+'/ner_dataset.csv', encoding="latin1")

In [3]:
len(df)

1048575

In [4]:
df

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O
...,...,...,...,...
1048570,NaN,they,PRP,O
1048571,NaN,responded,VBD,O
1048572,NaN,to,TO,O
1048573,NaN,the,DT,O


# preprocessing

In [5]:
# Step 1: Forward fill the 'Sentence #' column to associate each token with its sentence
df['Sentence #'] = df['Sentence #'].ffill()

# Step 2, 3 & 4: Group by 'Sentence #' and aggregate 'Word' and 'Tag' into lists
sentences_df = df.groupby('Sentence #').agg({
    'Word': list,
    'Tag': list
}).reset_index()

# Rename columns for clarity
sentences_df.columns = ['Sentence_Index', 'Sentence', 'Tags']

# Step 5: Verify the reconstruction
print(f"Total sentences reconstructed: {len(sentences_df)}")
print("Sample reconstructed sentence:")
print(sentences_df.head())


Total sentences reconstructed: 47959
Sample reconstructed sentence:
    Sentence_Index                                           Sentence  \
0      Sentence: 1  [Thousands, of, demonstrators, have, marched, ...   
1     Sentence: 10  [Iranian, officials, say, they, expect, to, ge...   
2    Sentence: 100  [Helicopter, gunships, Saturday, pounded, mili...   
3   Sentence: 1000  [They, left, after, a, tense, hour-long, stand...   
4  Sentence: 10000  [U.N., relief, coordinator, Jan, Egeland, said...   

                                                Tags  
0  [O, O, O, O, O, O, B-geo, O, O, O, O, O, B-geo...  
1  [B-gpe, O, O, O, O, O, O, O, O, O, O, O, O, O,...  
2  [O, O, B-tim, O, O, O, O, O, B-geo, O, O, O, O...  
3                  [O, O, O, O, O, O, O, O, O, O, O]  
4  [B-geo, O, O, B-per, I-per, O, B-tim, O, B-geo...  


In [6]:
def join_tokens(tokens):
    if isinstance(tokens, list):
        return ' '.join(map(str, tokens))
    return tokens

def get_entities_with_offsets(sentence, tags):
    """Extracts entities with start and end character positions."""
    words = sentence.split()
    entities = []
    char_cursor = 0
    current_entity = None
    current_start = 0

    for word, tag in zip(words, tags):
        if tag.startswith('B-'):
            if current_entity:
                # End previous entity
                entities.append((current_start, char_cursor - 1, current_entity))
            current_entity = tag[2:]
            current_start = char_cursor
        elif tag.startswith('I-'):
            if current_entity is None: # Handle I- without a B-
                current_entity = tag[2:]
                current_start = char_cursor
        else: # 'O' tag
            if current_entity:
                entities.append((current_start, char_cursor - 1, current_entity))
                current_entity = None

        char_cursor += len(word) + 1 # +1 for the space

    if current_entity:
        entities.append((current_start, char_cursor - 1, current_entity))

    return entities

# 1. Join tokens into sentences (if not already done)
sentences_df['Sentence'] = sentences_df['Sentence'].apply(join_tokens)

# 2. Extract entities with offsets
sentences_df['Entities'] = sentences_df.apply(lambda x: get_entities_with_offsets(x['Sentence'], x['Tags']), axis=1)

print("Reconstruction and entity extraction complete.")
print(sentences_df[['Sentence', 'Entities']].head())

Reconstruction and entity extraction complete.
                                            Sentence  \
0  Thousands of demonstrators have marched throug...   
1  Iranian officials say they expect to get acces...   
2  Helicopter gunships Saturday pounded militant ...   
3  They left after a tense hour-long standoff wit...   
4  U.N. relief coordinator Jan Egeland said Sunda...   

                                            Entities  
0    [(48, 54, geo), (77, 81, geo), (111, 118, gpe)]  
1      [(0, 7, gpe), (87, 96, tim), (108, 112, org)]  
2  [(20, 28, tim), (62, 69, geo), (97, 104, org),...  
3                                                 []  
4  [(0, 4, geo), (24, 35, per), (41, 47, tim), (5...  


In [7]:
train = []
for i in range(len(sentences_df)):
    d = dict()
    d2 = dict()
    sentence = sentences_df.at[i, 'Sentence']
    entities = sentences_df.at[i, 'Entities']
    d['text'] = sentence
    d2['entities'] = entities
    d['labels'] = d2
    train.append(d)

In [8]:
# Use a subset of the data for faster training
train = train[:1000]
len(train)

1000

# training

In [9]:
# Import necessary libraries
from transformers import BertTokenizer, BertForTokenClassification
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Updated entity types to match the actual labels in the dataset
# Common tags in this corpus: geo, gpe, per, org, tim, art, nat, eve
raw_tags = ['geo', 'gpe', 'per', 'org', 'tim', 'art', 'nat', 'eve']
entity_types = ["O"]
for tag in raw_tags:
    entity_types.append(f"B-{tag}")
    entity_types.append(f"I-{tag}")

num_labels = len(entity_types)

# Load pre-trained BERT model and tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForTokenClassification.from_pretrained('bert-base-uncased', num_labels=num_labels).to(device)

batch_size = 32
learning_rate = 5e-5
train_dataset_sample = train

def tokenize_and_format_data(dataset, tokenizer):
    tokenized_data = []
    for sample in dataset:
        text = sample["text"]
        entities = sample["labels"]["entities"]

        # Encode the text using the tokenizer
        encoding = tokenizer(text, truncation=True, padding='max_length', max_length=128)
        input_ids = encoding['input_ids']
        labels = ['O'] * len(input_ids)

        # Alignment logic
        for start, end, entity_type in entities:
            prefix_tokens = tokenizer.tokenize(text[:start])
            entity_tokens = tokenizer.tokenize(text[start:end])

            start_token_idx = len(prefix_tokens) + 1
            end_token_idx = start_token_idx + len(entity_tokens)

            if start_token_idx < len(labels):
                labels[start_token_idx] = f"B-{entity_type}"
                for i in range(start_token_idx + 1, min(end_token_idx, len(labels))):
                    labels[i] = f"I-{entity_type}"

        # Map label strings to IDs - now matches correctly
        label_ids = [entity_types.index(l) if l in entity_types else entity_types.index('O') for l in labels]

        tokenized_data.append({
            'input_ids': input_ids,
            'labels': label_ids
        })

    return TensorDataset(
        torch.tensor([item['input_ids'] for item in tokenized_data]),
        torch.tensor([item['labels'] for item in tokenized_data])
    )


# Prepare data and fine-tune
train_data = tokenize_and_format_data(train_dataset_sample, tokenizer)
train_dataloader = DataLoader(train_data, batch_size=batch_size)

optimizer = AdamW(model.parameters(), lr=learning_rate)
num_epochs = 10

for epoch in tqdm(range(num_epochs), desc="Training epochs"):
    model.train()
    for batch in train_dataloader:
        inputs, labels = [x.to(device) for x in batch]
        outputs = model(inputs, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

model.save_pretrained('fine_tuned_ner_model')
print("Training complete with corrected labels.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete with corrected labels.


# predicting

In [16]:
sample_text = train[0]["text"]
print(sample_text)
print()

# Tokenize the sample text
inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, padding='max_length', max_length=128)

# Move model and inputs to the appropriate device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Set model to evaluation mode
model.eval()

# Perform inference
with torch.no_grad():
    outputs = model(**inputs)

# Get predictions (argmax over the logits)
logits = outputs.logits
predictions = torch.argmax(logits, dim=2)
predicted_label_ids = predictions[0].cpu().numpy()

# Map tokens back to their predicted labels
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
results = []

for token, label_id in zip(tokens, predicted_label_ids):
    label = entity_types[label_id]
    # Ignore special tokens and 'O' (Outside) tags
    if label != "O" and token not in ["[CLS]", "[SEP]", "[PAD]"]:
        results.append((token, label))

# Display findings
if results:
    print("Detected Entities:")
    for token, label in results:
        print(f"{token}: {label}")
else:
    print("No entities were detected with the current model state.")

Thousands of demonstrators have marched through London to protest the war in Iraq and demand the withdrawal of British troops from that country .

Detected Entities:
london: B-geo
iraq: B-geo
british: B-gpe


In [15]:
sample_text = train[1]["text"]
print(sample_text)
print()

# Tokenize the sample text
inputs = tokenizer(sample_text, return_tensors="pt", truncation=True, padding='max_length', max_length=128)

# Move model and inputs to the appropriate device (GPU if available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
inputs = {k: v.to(device) for k, v in inputs.items()}

# Set model to evaluation mode
model.eval()

# Perform inference
with torch.no_grad():
    outputs = model(**inputs)

# Get predictions (argmax over the logits)
logits = outputs.logits
predictions = torch.argmax(logits, dim=2)
predicted_label_ids = predictions[0].cpu().numpy()

# Map tokens back to their predicted labels
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
results = []

for token, label_id in zip(tokens, predicted_label_ids):
    label = entity_types[label_id]
    # Ignore special tokens and 'O' (Outside) tags
    if label != "O" and token not in ["[CLS]", "[SEP]", "[PAD]"]:
        results.append((token, label))

# Display findings
if results:
    print("Detected Entities:")
    for token, label in results:
        print(f"{token}: {label}")
else:
    print("No entities were detected with the current model state.")

Iranian officials say they expect to get access to sealed sensitive parts of the plant Wednesday , after an IAEA surveillance system begins functioning .

Detected Entities:
iranian: B-gpe
wednesday: B-tim
ia: B-org
##ea: I-org
